# LangGraph: a hands-on implementation

**Track:** Frameworks · **Guide:** [`055-langgraph.md`](../guides/concepts/055-langgraph.md)

LangGraph models an agent or workflow as state transitions between named nodes, with explicit edges and stop conditions.

> **Intuition:** A flowchart with typed shared state makes branching, retries, and human approval visible.

This lesson builds a small working example in stages: prepare inputs, implement
the core idea, run it on a realistic toy case, and inspect its output and limits.
It runs offline unless a cell is explicitly marked as an optional provider call.

```mermaid
flowchart LR
    A[Prepare inputs] --> B[Implement LangGraph]
    B --> C[Run on example]
    C --> D[Evaluate and inspect]
```

**Case study:** Pause an assistant before a side effect, request human approval, then resume from saved state.

Run the cells from top to bottom. To install the notebook environment, follow
[`guides/SETUP.md`](../guides/SETUP.md). The toy data and implementation are for
learning, not a production quality, safety, or security guarantee.


## 1. Define explicit workflow state

LangGraph models a workflow as nodes that transform shared state and edges that route execution.


In [ ]:
from typing import TypedDict


class State(TypedDict):
    question: str
    evidence: list[str]
    approved: bool
    answer: str

initial_state: State = {
    "question": "What is the return window?",
    "evidence": ["Returns are accepted within 30 days."],
    "approved": False,
    "answer": "",
}

## 2. Implement nodes and conditional transitions

The small runner below makes nodes and transitions visible without requiring network access or framework setup.


In [ ]:
def draft_node(state):
    if not state["evidence"]:
        return {**state, "answer": "I don't know."}
    return {**state, "answer": state["evidence"][0]}

def route_after_draft(state):
    return "finish" if state["approved"] else "human_review"

def run_graph(state):
    state = draft_node(state)
    route = route_after_draft(state)
    if route == "human_review":
        state = {**state, "answer": f"REVIEW REQUIRED: {state['answer']}"}
    return state, route

## 3. Run the graph and inspect the branch

For a real LangGraph workflow, add checkpointing and resume at the approval node; `examples/integrations/langgraph_example.py` demonstrates its optional framework API.


In [ ]:
result, route = run_graph(initial_state)
print("route:", route)
print("state:", result)
assert route == "human_review"

## 4. Optionally run a LangGraph implementation

Install the `langgraph` extra to run the same approval branch with the framework's state graph.


In [ ]:
RUN_LANGGRAPH = False
if RUN_LANGGRAPH:
    from examples.integrations.langgraph_example import build_workflow

    workflow_result = build_workflow().invoke({
        "approved": False, "action": "send_email", "status": "new"
    })
    print(workflow_result)
else:
    print("Framework call skipped; the local state graph above runs offline.")

## Wrap-up and next steps

- Identify what was learned, stored, retrieved, or generated in this implementation.
- Change an input and observe how the output changes.
- Test one failure case and decide when the system should abstain or ask for help.
- Replace toy data/components only after you can evaluate the change against a
  held-out or representative test set.

Continue with the [concept guide](../guides/concepts/055-langgraph.md) and the
[learning roadmap](../guides/ROADMAP.md).
